# Exercise 4A - Stress-Testing a Visual Classifier

**Goal:** measure how a fixed image classifier degrades under
controlled common-corruption families and increasing severity.

This notebook follows the evaluation logic of common-corruption
benchmarks:

$x' = T_{p,s}(x), \qquad \Delta M_{p,s} = M_{clean} - M_{p,s}$

where `p` is the corruption family and `s` is the severity.

The implementations here are simplified teaching approximations,
not exact CIFAR-10-C benchmark code.

**Difficulty**

- 🟢 implement local image operations
- 🟡 build the structured evaluation grid
- 🔴 optional analysis and extensions

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import os
import sys
import tarfile
import requests
from tqdm.auto import tqdm

os.environ.setdefault(
    "MPLCONFIGDIR",
    str((Path.cwd() / ".matplotlib_cache").resolve()),
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance, ImageFilter

import torch
from torchvision import datasets
from torchvision.datasets.utils import check_integrity

repo_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().parent.parent
    if Path.cwd().name == "solutions"
    else Path.cwd()
)
sys.path.append(str(repo_root / "src"))

from cv4is_robustness.data import make_cifar10_bundle
from cv4is_robustness.models import (
    TinyCIFARNet,
    count_parameters,
    evaluate_loader,
    fit_model,
    get_device,
    load_checkpoint,
    save_checkpoint,
    set_seed,
)
from cv4is_robustness.evaluation import (
    collect_failure_examples,
    evaluate_corrupted_subset,
)
from cv4is_robustness.visualization import (
    plot_metric_heatmap,
    plot_robustness_curves,
    show_corruption_grid,
    show_failure_examples,
    show_image_batch,
)

SEED = 42
set_seed(SEED)

device = get_device()
print("Repository root:", repo_root)
print("Device:", device)

## Workflow and runtime controls

The model is deliberately compact. Training is not the learning goal
today, so the training loop is provided.

On a slower CPU, reduce `TRAIN_SAMPLES`, `EPOCHS`, or
`STRESS_TEST_SAMPLES`. Keep the same fixed test indices when comparing
corruption families.

In [ ]:
TRAIN_SAMPLES = 15_000
VAL_SAMPLES = 2_000
TEST_SAMPLES = 2_000
BATCH_SIZE = 128
EPOCHS = 10

STRESS_TEST_SAMPLES = 800
SEVERITIES = [0, 1, 2, 3, 4, 5]

data_root = repo_root / "data"
artifact_root = repo_root / "artifacts"
checkpoint_path = artifact_root / "tiny_cifar10_checkpoint.pt"

print("Checkpoint:", checkpoint_path)

## Part A - Download and inspect CIFAR-10

CIFAR-10 contains real RGB images from ten object classes. It is not an
industrial inspection benchmark, but it provides a compact and reproducible
image-classification task for this lab.

The following cell downloads the original dataset archive from Sciebo,
verifies it using TorchVision's official hash, extracts it, creates the
train/validation/test loaders, and displays example images.

**NOTE:** CIFAR-10 images have a resolution of only **32 × 32 pixels**. They may therefore
appear visibly pixelated when displayed in the notebook. This is a property of
the original dataset, not a download or preprocessing error.

In [ ]:
SCIEBO_URL = "https://ruhr-uni-bochum.sciebo.de/s/j89FGFzH38MQpDp"

data_root = Path(data_root)
data_root.mkdir(parents=True, exist_ok=True)

archive_path = data_root / datasets.CIFAR10.filename


def cifar10_is_ready() -> bool:
    """Check the hashes of all extracted CIFAR-10 files."""
    try:
        datasets.CIFAR10(
            root=data_root,
            train=True,
            download=False,
        )
        return True
    except RuntimeError:
        return False


def download_from_sciebo(
    share_url: str,
    output_path: Path,
) -> None:
    """Download a public Sciebo file share through WebDAV."""
    parsed = urlparse(share_url)
    token = parsed.path.rstrip("/").split("/")[-1]
    webdav_url = (
        f"{parsed.scheme}://{parsed.netloc}/public.php/webdav/"
    )

    temporary_path = output_path.with_suffix(
        output_path.suffix + ".part"
    )
    temporary_path.unlink(missing_ok=True)

    with requests.get(
        webdav_url,
        auth=(token, ""),  # Public share: no password
        stream=True,
        timeout=(15, 120),
    ) as response:
        response.raise_for_status()

        total_bytes = int(
            response.headers.get("content-length", 0)
        )

        with temporary_path.open("wb") as file, tqdm(
            total=total_bytes or None,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc="Downloading CIFAR-10",
        ) as progress:
            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    file.write(chunk)
                    progress.update(len(chunk))

    temporary_path.replace(output_path)


if not cifar10_is_ready():
    print("CIFAR-10 not found. Downloading from Sciebo...")

    if not archive_path.exists():
        download_from_sciebo(
            SCIEBO_URL,
            archive_path,
        )

    if not check_integrity(
        archive_path,
        md5=datasets.CIFAR10.tgz_md5,
    ):
        archive_path.unlink(missing_ok=True)
        raise RuntimeError(
            "The downloaded archive failed the official "
            "CIFAR-10 MD5 verification."
        )

    print("Archive verified. Extracting...")

    with tarfile.open(archive_path, "r:gz") as archive:
        destination = data_root.resolve()

        for member in archive.getmembers():
            member_path = (data_root / member.name).resolve()

            try:
                member_path.relative_to(destination)
            except ValueError as error:
                raise RuntimeError(
                    f"Unsafe archive path: {member.name}"
                ) from error

        archive.extractall(data_root)

    if not cifar10_is_ready():
        raise RuntimeError(
            "The extracted CIFAR-10 files failed "
            "TorchVision's integrity checks."
        )

    archive_path.unlink(missing_ok=True)
    print("Extraction and integrity checks completed.")

else:
    print("CIFAR-10 already exists and passed integrity checks.")


bundle = make_cifar10_bundle(
    data_root=data_root,
    train_samples=TRAIN_SAMPLES,
    val_samples=VAL_SAMPLES,
    test_samples=TEST_SAMPLES,
    batch_size=BATCH_SIZE,
    seed=SEED,
    num_workers=0,
)

class_names = bundle.class_names

print("Classes:", class_names)
print("Train batches:", len(bundle.train_loader))
print("Validation batches:", len(bundle.val_loader))
print("Test batches:", len(bundle.test_loader))

preview = [
    bundle.raw_test_dataset[index]
    for index in bundle.test_indices[:12]
]

images = [image for image, _ in preview]
labels = [label for _, label in preview]

fig, axes = plt.subplots(
    2,
    6,
    figsize=(8, 3),
)

for ax, image, label in zip(axes.flat, images, labels):
    ax.imshow(image, interpolation="none")
    ax.set_title(class_names[label], fontsize=8)
    ax.axis("off")

print("NOTE: CIFAR-10 images are 32x32 and may appear pixelated in the notebook")
plt.tight_layout()
plt.show()

## Part B - Train or load a fixed baseline

Robustness comparisons require a **fixed model**. We therefore train
once, save the best validation checkpoint, and reuse it for every
corruption condition.

Do not tune the model separately for each corruption on the final
test set.

In [ ]:
model = TinyCIFARNet(num_classes=len(class_names)).to(device)
print("Trainable parameters:", f"{count_parameters(model):,}")

if checkpoint_path.exists():
    checkpoint = load_checkpoint(
        model,
        checkpoint_path,
        device,
    )
    print("Loaded existing checkpoint.")
    print("Saved config:", checkpoint.get("config", {}))
else:
    print("No checkpoint found. Training the compact baseline...")
    history = fit_model(
        model,
        bundle.train_loader,
        bundle.val_loader,
        device=device,
        epochs=EPOCHS,
        learning_rate=2e-3,
    )

    save_checkpoint(
        model,
        checkpoint_path,
        class_names=class_names,
        config={
            "seed": SEED,
            "train_samples": TRAIN_SAMPLES,
            "val_samples": VAL_SAMPLES,
            "test_samples": TEST_SAMPLES,
            "epochs": EPOCHS,
        },
    )
    print("Saved checkpoint to:", checkpoint_path)

clean_loader_metrics = evaluate_loader(
    model,
    bundle.test_loader,
    device,
)
print("Clean test metrics:", clean_loader_metrics)

## Part C - Implement severity-controlled corruptions

A useful corruption function must:

1. accept an image and a severity,
2. map severity to a documented parameter,
3. preserve image size and RGB format,
4. clip pixels to a valid range,
5. preserve the intended semantic label over the chosen severity range.

We implement four families:

- Gaussian noise,
- defocus-like blur,
- pixelation,
- brightness / overexposure.

The brightness function is provided as an example.

### Task C1 - Gaussian noise 🟢

Add zero-mean Gaussian noise with severity-dependent standard
deviation. Use the supplied random generator so repeated experiments
are reproducible.

### Task C2 - Defocus-like blur 🟢

Use a severity-dependent blur radius.

### Task C3 - Pixelation 🟢

Downsample and then restore the original image size using nearest
neighbour interpolation.

In [ ]:
NOISE_STD = [0.00, 0.03, 0.06, 0.10, 0.15, 0.22]
BLUR_RADIUS = [0.0, 0.4, 0.8, 1.2, 1.8, 2.5]
PIXELATE_SCALE = [1.00, 0.85, 0.70, 0.55, 0.40, 0.25]
BRIGHTNESS_FACTOR = [1.00, 1.10, 1.25, 1.45, 1.70, 2.00]


def validate_severity(severity: int) -> None:
    if severity not in range(6):
        raise ValueError("severity must be an integer from 0 to 5")


def gaussian_noise(
    image: Image.Image,
    severity: int,
    rng: np.random.Generator,
) -> Image.Image:
    validate_severity(severity)

    # TODO:
    # 1. Convert the image to float32 in [0, 1].
    # 2. Draw Gaussian noise with NOISE_STD[severity].
    # 3. Add and clip to [0, 1].
    # 4. Convert back to an RGB PIL image.
    raise NotImplementedError("Implement gaussian_noise")


def defocus_blur(
    image: Image.Image,
    severity: int,
    rng: np.random.Generator,
) -> Image.Image:
    del rng
    validate_severity(severity)

    # TODO:
    # Apply PIL.ImageFilter.GaussianBlur with the radius
    # stored in BLUR_RADIUS[severity].
    raise NotImplementedError("Implement defocus_blur")


def pixelate(
    image: Image.Image,
    severity: int,
    rng: np.random.Generator,
) -> Image.Image:
    del rng
    validate_severity(severity)

    # TODO:
    # 1. Return a copy for severity 0.
    # 2. Downsample using PIXELATE_SCALE[severity].
    # 3. Restore the original size with nearest-neighbour
    #    interpolation.
    raise NotImplementedError("Implement pixelate")


def brightness_shift(
    image: Image.Image,
    severity: int,
    rng: np.random.Generator,
) -> Image.Image:
    del rng
    validate_severity(severity)
    return ImageEnhance.Brightness(image).enhance(
        BRIGHTNESS_FACTOR[severity]
    )


CORRUPTIONS = {
    "gaussian_noise": gaussian_noise,
    "defocus_blur": defocus_blur,
    "pixelate": pixelate,
    "brightness": brightness_shift,
}

In [ ]:
# Sanity checks. Run after implementing the three functions.
test_image, _ = bundle.raw_test_dataset[bundle.test_indices[0]]

for corruption_name, corruption_fn in CORRUPTIONS.items():
    output = corruption_fn(
        test_image,
        severity=3,
        rng=np.random.default_rng(123),
    )

    assert isinstance(output, Image.Image)
    assert output.mode == "RGB"
    assert output.size == test_image.size

print("Corruption interface checks passed.")

In [ ]:
example_image, example_label = bundle.raw_test_dataset[
    bundle.test_indices[3]
]
print("Example class:", class_names[example_label])

show_corruption_grid(
    example_image,
    CORRUPTIONS,
    severities=SEVERITIES,
)

### Label-preservation check

Look at the strongest severity for each family.

- Is the original class still recognizable?
- At which point does the corruption change the semantic content
  rather than merely degrading acquisition quality?
- Would the same severity be label-preserving for a tiny surface
  defect or a small OCR character?

Robustness results are only meaningful over a defensible
label-preserving range.

## Part D - Prediction confidence and entropy

For class probabilities \(p_k\), predictive entropy is:

$H(p) = -\sum_k p_k \log p_k$

### Task D1 - Implement predictive entropy 🟢

The function should accept a tensor of shape `(N, C)` and return
one entropy value per sample.

In [ ]:
def predictive_entropy(probabilities: torch.Tensor) -> torch.Tensor:
    # TODO:
    # 1. Clamp probabilities away from zero.
    # 2. Compute -sum(p * log(p)) across the class dimension.
    raise NotImplementedError("Implement predictive_entropy")


test_probabilities = torch.tensor(
    [
        [0.9, 0.1],
        [0.5, 0.5],
    ],
    dtype=torch.float32,
)

test_entropy = predictive_entropy(test_probabilities)
print("Entropy values:", test_entropy.tolist())

assert test_entropy.shape == (2,)
assert test_entropy[1] > test_entropy[0]
print("Entropy test passed.")

## Part E - Run the corruption-by-severity evaluation grid

For each corruption family:

1. evaluate severity 0 to 5,
2. repeat stochastic Gaussian noise twice,
3. use the same fixed test subset,
4. record accuracy, macro-F1, confidence, and entropy.

### Task E1 - Complete the evaluation loop 🟡

Use `evaluate_corrupted_subset(...)` and append one dictionary per
run to `results`.

In [ ]:
stress_indices = bundle.test_indices[:STRESS_TEST_SAMPLES]
results = []

for corruption_name, corruption_fn in CORRUPTIONS.items():
    repeats = 2 if corruption_name == "gaussian_noise" else 1

    for severity in SEVERITIES:
        for repeat in range(repeats):
            # TODO:
            # 1. Call evaluate_corrupted_subset(...).
            # 2. Append corruption, severity, repeat, accuracy,
            #    macro-F1, mean confidence, and mean entropy.
            raise NotImplementedError(
                "Complete the corruption evaluation loop"
            )

results_df = pd.DataFrame(results)
results_df.head()

In [ ]:
summary_df = (
    results_df
    .groupby(["corruption", "severity"], as_index=False)
    .agg(
        accuracy=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        macro_f1=("macro_f1", "mean"),
        mean_confidence=("mean_confidence", "mean"),
        mean_entropy=("mean_entropy", "mean"),
    )
)

summary_df["accuracy_std"] = summary_df["accuracy_std"].fillna(0.0)

clean_accuracy = float(
    summary_df.loc[
        summary_df["severity"] == 0,
        "accuracy",
    ].mean()
)
clean_macro_f1 = float(
    summary_df.loc[
        summary_df["severity"] == 0,
        "macro_f1",
    ].mean()
)

summary_df["accuracy_drop"] = clean_accuracy - summary_df["accuracy"]
summary_df["macro_f1_drop"] = clean_macro_f1 - summary_df["macro_f1"]

print("Clean accuracy:", f"{clean_accuracy:.3f}")
print("Clean macro-F1:", f"{clean_macro_f1:.3f}")
display(summary_df.round(3))

In [ ]:
plot_robustness_curves(summary_df, metric="accuracy")
plot_robustness_curves(summary_df, metric="macro_f1")
plot_robustness_curves(summary_df, metric="mean_confidence")
plot_robustness_curves(summary_df, metric="mean_entropy")

plot_metric_heatmap(summary_df, metric="accuracy")
plot_metric_heatmap(summary_df, metric="accuracy_drop")

## Part F - Scalar robustness summaries

A curve is more informative than one number, but scalar summaries
are useful for model comparison.

We use a normalized trapezoidal area under the accuracy-versus-
severity curve:

$R_p = \int_0^1 A_p(s)\,ds$

where severity is normalized from 0 to 1.

We then report:

- `R_avg`: average across corruption families,
- `R_min`: worst-family robustness.

### Task F1 - Implement normalized robustness AUC 🟡

In [ ]:
def normalized_robustness_auc(group: pd.DataFrame) -> float:
    # TODO:
    # 1. Sort by severity.
    # 2. Normalize severity to [0, 1].
    # 3. Integrate accuracy with np.trapz.
    raise NotImplementedError(
        "Implement normalized_robustness_auc"
    )


robustness_rows = []

for corruption_name, group in summary_df.groupby("corruption"):
    robustness_rows.append(
        {
            "corruption": corruption_name,
            "robustness_auc": normalized_robustness_auc(group),
            "worst_accuracy": float(group["accuracy"].min()),
            "max_accuracy_drop": float(group["accuracy_drop"].max()),
        }
    )

robustness_df = pd.DataFrame(robustness_rows).sort_values(
    "robustness_auc",
    ascending=False,
)

R_avg = float(robustness_df["robustness_auc"].mean())
R_min = float(robustness_df["robustness_auc"].min())

display(robustness_df.round(3))
print("Average robustness R_avg:", f"{R_avg:.3f}")
print("Worst-family robustness R_min:", f"{R_min:.3f}")

### Illustrative acceptance criterion

The next cell uses a **teaching threshold** of at least 80% of
clean accuracy. This is not a valid industrial acceptance
criterion by itself.

In a real system, the threshold should come from operational
consequences such as missed defects, false rejects, rework,
downtime, or safety risk, and should be defined before final test
inspection.

In [ ]:
illustrative_threshold = 0.80 * clean_accuracy

threshold_table = summary_df.copy()
threshold_table["acceptable"] = (
    threshold_table["accuracy"] >= illustrative_threshold
)

first_failure = (
    threshold_table.loc[~threshold_table["acceptable"]]
    .sort_values(["corruption", "severity"])
    .groupby("corruption", as_index=False)
    .first()
)

print(
    "Illustrative threshold:",
    f"{illustrative_threshold:.3f}",
)
display(
    first_failure[
        ["corruption", "severity", "accuracy", "accuracy_drop"]
    ].round(3)
)

## Part G - Inspect corruption-induced failures

Aggregate curves can hide how a model fails. We now search for
images that are:

- correctly classified when clean,
- incorrectly classified after the strongest corruption.

Inspecting failures helps distinguish plausible acquisition
degradation from transformations that destroy the label.

In [ ]:
worst_family = (
    robustness_df
    .sort_values("robustness_auc")
    .iloc[0]["corruption"]
)
worst_function = CORRUPTIONS[worst_family]

examples = collect_failure_examples(
    model=model,
    raw_dataset=bundle.raw_test_dataset,
    indices=stress_indices,
    corruption_fn=worst_function,
    severity=5,
    eval_transform=bundle.eval_transform,
    class_names=class_names,
    device=device,
    max_examples=6,
    seed=SEED,
)

print("Worst family by AUC:", worst_family)
show_failure_examples(examples)

## Final interpretation

Answer in 8-12 sentences:

1. Which corruption family causes the fastest degradation?
2. Is degradation monotonic with severity?
3. Does mean confidence decrease at the same rate as accuracy?
4. Does entropy reliably identify every failure?
5. Which result is hidden by the average robustness score?
6. Is severity 5 label-preserving for every image?
7. Why is clean accuracy insufficient for deployment?
8. What corruption families would matter for an industrial
   inspection camera?

In [ ]:
answer = """
TODO: Write your interpretation here.
"""
print(answer)

## Optional extensions 🔴

- Add motion blur or JPEG compression.
- Increase Gaussian-noise repeats and report confidence intervals.
- Compare two model architectures with the same corruption grid.
- Train with corruption augmentation, then repeat the untouched test
  protocol.
- Replace classification accuracy with class-specific recall.
- For an industrial defect task, report false accept and false reject
  rates instead of only aggregate accuracy.
- Test compound corruptions such as blur plus low brightness.